# EpidemicWatch — Phase 4: Federated Learning (Flower + LoRA + FedAvg/FedProx)

Federates the Phase 3 centralized baseline (`Bio_ClinicalBERT` + LoRA) across your
3 simulated district hospitals using the **Flower** framework. Only LoRA adapter +
classification-head weights are ever transmitted between clients and server — the
frozen Bio_ClinicalBERT base model never moves, which is the whole point of doing
LoRA-federation instead of full-model federation.

**Centralized baseline to beat (from Phase 3, augmented run):** 93.7% accuracy,
94.3% macro-F1 on the pooled test set.

**Experiments run in this notebook, all under the same conditions otherwise:**
1. FedAvg on the **non-IID** hospital split (μ=0)
2. FedProx on the **non-IID** split, μ=0.01
3. FedProx on the **non-IID** split, μ=0.1
4. FedAvg on the **IID** split (control, should converge fastest/highest)

> **A note on the Flower API:** this notebook uses `flwr.simulation.run_simulation`
> with the `ClientApp`/`ServerApp` pattern. Flower's docs currently recommend the
> `flwr run` CLI workflow instead, but that requires a multi-file project
> structure (`pyproject.toml` etc.) that doesn't fit a single Colab notebook.
> `run_simulation` still works correctly (just prints a deprecation warning you
> can ignore) and is the standard way to run Flower simulations from a notebook.

> Run this in Google Colab with a **GPU runtime**. Federated training runs the
> full Phase-3-style training loop once per hospital per round, several times
> over — this is the most compute-heavy notebook in the project.

## 0. Setup

In [ ]:
!pip install -q peft transformers accelerate flwr ray
# Colab pre-installs an old torchao (quantization library) that peft checks for
# and hard-fails on if present-but-too-old, even though we never use it here.
# Removing it entirely avoids the version-check crash.
!pip uninstall -y -q torchao 2>/dev/null || true

import os
import json
import random
import time
import zipfile
from collections import OrderedDict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
import seaborn as sns

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from peft.utils import get_peft_model_state_dict, set_peft_model_state_dict

from flwr.client import ClientApp, NumPyClient
from flwr.common import Context, ndarrays_to_parameters
from flwr.server import ServerApp, ServerAppComponents, ServerConfig
from flwr.server.strategy import FedAvg
from flwr.simulation import run_simulation

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# ---- Config ---------------------------------------------------------------
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
DATA_DIR = "epidemicwatch_data"
MAX_LENGTH = 128
BATCH_SIZE = 16
LOCAL_LR = 1e-4
LOCAL_EPOCHS = 2          # local epochs per client per round
NUM_ROUNDS = 8
NUM_HOSPITALS = 3
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.1
CLASSIFIER_DROPOUT = 0.3
CENTRALIZED_BASELINE = {"accuracy": 0.9371, "f1_macro": 0.9430}  # from Phase 3 (augmented run)
OUTPUT_DIR = "phase4_federated_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# NOTE: client-side compute is CPU-only per Ray actor by default in this
# notebook (num_gpus=0.0) even on a GPU runtime, because 3 clients sharing one
# GPU via Ray's simulation backend can be unreliable in Colab. If you have a
# beefier GPU and want to try client-side GPU sharing, change
# CLIENT_NUM_GPUS below to e.g. 0.3 -- test it on 1 round first.
CLIENT_NUM_GPUS = 0.0
CLIENT_RESOURCES = {"num_cpus": 1, "num_gpus": CLIENT_NUM_GPUS}

print("Config set.")

## 1. Load Data (hospital shards + pooled val/test) and Tokenizer

Loads Phase 2's non-IID and IID hospital shards, plus builds the pooled val/test
sets (same construction as Phase 3) for server-side round-by-round evaluation
and final test-set comparison against the centralized baseline.

In [ ]:
def data_root_is_valid(root):
    # True only if root actually contains the expected hospital shard files,
    # not just a folder with the right name.
    probe = os.path.join(root, "non_iid", "hospital_0", "train.csv")
    return os.path.isfile(probe)

def find_data_root(start=".", max_depth=3):
    # Search a few levels deep for a directory that actually has the expected
    # structure -- handles the common case where a zip was extracted with an
    # extra nesting level (e.g. epidemicwatch_data/epidemicwatch_data/...).
    import glob
    for depth in range(max_depth + 1):
        pattern = os.path.join(start, *(["*"] * depth), "non_iid", "hospital_0", "train.csv")
        matches = glob.glob(pattern)
        if matches:
            return os.path.dirname(os.path.dirname(os.path.dirname(matches[0])))
    return None

resolved_root = find_data_root()

if resolved_root is None:
    from google.colab import files
    print(f"Could not find a valid '{DATA_DIR}' structure anywhere in the current "
          f"directory tree. Please upload epidemicwatch_data.zip from Phase 2.")
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    extract_target = DATA_DIR
    with zipfile.ZipFile(zip_name, "r") as z:
        z.extractall(extract_target)
    resolved_root = find_data_root()
    if resolved_root is None:
        print("Still couldn't locate the expected structure after extraction. Contents found:")
        for dirpath, dirnames, filenames in os.walk(extract_target):
            depth = dirpath.replace(extract_target, "").count(os.sep)
            if depth <= 3:
                print("  " * depth + os.path.basename(dirpath) + "/")
        raise FileNotFoundError(
            "Expected 'non_iid/hospital_0/train.csv' somewhere under the uploaded zip "
            "but didn't find it -- check the printed structure above and adjust DATA_DIR manually."
        )

DATA_DIR = resolved_root
print(f"Using data root: '{DATA_DIR}/' (validated: hospital shard files found).")

In [ ]:
def load_hospital_split(hospital_idx, split_name, source, data_dir=DATA_DIR):
    path = os.path.join(data_dir, source, f"hospital_{hospital_idx}", f"{split_name}.csv")
    return pd.read_csv(path)

def load_pooled_split(split_name, source, data_dir=DATA_DIR):
    dfs = [load_hospital_split(h, split_name, source, data_dir) for h in range(NUM_HOSPITALS)]
    return pd.concat(dfs, ignore_index=True)

# Categories / label maps -- built from the pooled non-IID train set (covers all classes)
_ref_df = load_pooled_split("train", "non_iid")
CATEGORIES = sorted(_ref_df["syndrome_category"].unique())
label2id = {c: i for i, c in enumerate(CATEGORIES)}
id2label = {i: c for c, i in label2id.items()}
NUM_LABELS = len(CATEGORIES)
print("Categories:", CATEGORIES)

# Pooled val (for server-side round-by-round eval) and test (final comparison) sets
pooled_val_df = load_pooled_split("val", "non_iid")     # non_iid/iid val splits are the same underlying pooled data
pooled_test_df = load_pooled_split("test", "non_iid")
for d in (pooled_val_df, pooled_test_df):
    d["label_id"] = d["syndrome_category"].map(label2id)
print(f"Pooled val: {len(pooled_val_df)} | Pooled test: {len(pooled_test_df)}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class SymptomDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=MAX_LENGTH):
        self.texts = df["text"].tolist()
        self.labels = df["label_id"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True, padding="max_length",
                              max_length=self.max_length, return_tensors="pt")
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

pooled_val_loader = DataLoader(SymptomDataset(pooled_val_df, tokenizer), batch_size=BATCH_SIZE)
pooled_test_loader = DataLoader(SymptomDataset(pooled_test_df, tokenizer), batch_size=BATCH_SIZE)
print("Data ready.")

## 2. Model + LoRA \<-\> Flower Parameter Conversion Helpers

In [ ]:
def build_lora_model():
    base_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_LABELS, id2label=id2label, label2id=label2id,
        classifier_dropout=CLASSIFIER_DROPOUT,
    )
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        target_modules=["query", "value"], modules_to_save=["classifier"],
    )
    return get_peft_model(base_model, lora_config)


# Fixed key order for LoRA state dict -- confirmed identical across fresh model
# instances of the same architecture, which is required since Flower passes
# plain lists of ndarrays (not named dicts) between client and server.
_REFERENCE_KEYS = list(get_peft_model_state_dict(build_lora_model()).keys())
print(f"LoRA + classifier trainable tensors transmitted per round: {len(_REFERENCE_KEYS)}")

def get_lora_ndarrays(model):
    sd = get_peft_model_state_dict(model)
    return [sd[k].cpu().numpy() for k in _REFERENCE_KEYS]

def set_lora_ndarrays(model, ndarrays):
    sd = OrderedDict((k, torch.tensor(v)) for k, v in zip(_REFERENCE_KEYS, ndarrays))
    set_peft_model_state_dict(model, sd)

def transmitted_size_mb(ndarrays):
    return sum(arr.nbytes for arr in ndarrays) / 1e6

print(f"Approx. size transmitted per round per client: "
      f"{transmitted_size_mb(get_lora_ndarrays(build_lora_model())):.2f} MB")

## 3. Evaluation Helper (shared by server-side eval and final test-set eval)

In [ ]:
def evaluate_model(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(next(model.parameters()).device) for k, v in batch.items()}
            out = model(**batch)
            total_loss += out.loss.item() * batch["labels"].size(0)
            all_preds.extend(out.logits.argmax(dim=-1).cpu().numpy())
            all_labels.extend(batch["labels"].cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    _, _, f1_macro, _ = precision_recall_fscore_support(all_labels, all_preds, average="macro", zero_division=0)
    _, _, f1_weighted, _ = precision_recall_fscore_support(all_labels, all_preds, average="weighted", zero_division=0)
    return {
        "loss": total_loss / len(loader.dataset), "accuracy": acc,
        "f1_macro": f1_macro, "f1_weighted": f1_weighted,
        "preds": all_preds, "labels": all_labels,
    }

## 4. Flower Client

Each simulated hospital is one client. `fit()` optionally adds the FedProx
proximal term (controlled via `config["proximal_mu"]`, sent by the server each
round) — when μ=0 this is exactly plain FedAvg local training.

In [ ]:
class HospitalClient(NumPyClient):
    def __init__(self, hospital_idx, data_source):
        self.hospital_idx = hospital_idx
        train_df = load_hospital_split(hospital_idx, "train", data_source)
        train_df["label_id"] = train_df["syndrome_category"].map(label2id)
        self.train_loader = DataLoader(
            SymptomDataset(train_df, tokenizer), batch_size=BATCH_SIZE, shuffle=True
        )
        self.n_train = len(train_df)
        self.model = build_lora_model().to(DEVICE)

    def get_parameters(self, config):
        return get_lora_ndarrays(self.model)

    def fit(self, parameters, config):
        set_lora_ndarrays(self.model, parameters)
        mu = config.get("proximal_mu", 0.0)
        # Frozen snapshot of the GLOBAL params received this round, used as the
        # anchor point for the FedProx proximal term (no grad needed on these).
        global_tensors = [torch.tensor(p).to(DEVICE) for p in parameters]

        optimizer = torch.optim.AdamW(self.model.parameters(), lr=LOCAL_LR)
        self.model.train()
        last_loss = 0.0
        for _ in range(LOCAL_EPOCHS):
            for batch in self.train_loader:
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                optimizer.zero_grad()
                out = self.model(**batch)
                loss = out.loss
                if mu > 0:
                    local_sd = get_peft_model_state_dict(self.model)
                    prox_term = sum(
                        torch.sum((local_sd[k] - g) ** 2)
                        for k, g in zip(_REFERENCE_KEYS, global_tensors)
                    )
                    loss = loss + (mu / 2.0) * prox_term
                loss.backward()
                optimizer.step()
                last_loss = loss.item()

        return get_lora_ndarrays(self.model), self.n_train, {"train_loss": float(last_loss)}

    def evaluate(self, parameters, config):
        set_lora_ndarrays(self.model, parameters)
        self.model.to(DEVICE)
        metrics = evaluate_model(self.model, self.train_loader)  # quick local check, not used for reported results
        return float(metrics["loss"]), self.n_train, {"accuracy": float(metrics["accuracy"])}

## 5. Server-Side Evaluation + Simulation Runner

Server-side `evaluate_fn` runs the aggregated GLOBAL model on the pooled
validation set after every round — this is what produces the convergence
curves, and keeps the test set untouched until the very end.

In [ ]:
def run_federated_experiment(run_name, data_source, proximal_mu, num_rounds=NUM_ROUNDS):
    print(f"\n{'='*60}\nRun: {run_name} | data={data_source} | proximal_mu={proximal_mu}\n{'='*60}")

    round_history = []  # captured via closure inside evaluate_fn below
    eval_model_holder = {"model": None}  # lazily built once, reused across rounds
    init_arrs = get_lora_ndarrays(build_lora_model())  # computed once here, reused below AND inside server_fn

    def evaluate_fn(server_round, parameters, config):
        if eval_model_holder["model"] is None:
            eval_model_holder["model"] = build_lora_model().to(DEVICE)
        set_lora_ndarrays(eval_model_holder["model"], parameters)
        metrics = evaluate_model(eval_model_holder["model"], pooled_val_loader)
        round_history.append({
            "round": server_round, "val_loss": metrics["loss"],
            "val_accuracy": metrics["accuracy"], "val_f1_macro": metrics["f1_macro"],
        })
        print(f"  [round {server_round}] val_loss={metrics['loss']:.4f} "
              f"val_acc={metrics['accuracy']:.4f} val_f1_macro={metrics['f1_macro']:.4f}")
        return metrics["loss"], {"accuracy": metrics["accuracy"], "f1_macro": metrics["f1_macro"]}

    def client_fn(context: Context):
        hospital_idx = context.node_config["partition-id"]
        return HospitalClient(hospital_idx, data_source).to_client()

    def server_fn(context: Context):
        strategy = FedAvg(
            fraction_fit=1.0, fraction_evaluate=0.0,   # server-side evaluate_fn used instead of client-side eval
            min_fit_clients=NUM_HOSPITALS, min_available_clients=NUM_HOSPITALS,
            initial_parameters=ndarrays_to_parameters(init_arrs),
            evaluate_fn=evaluate_fn,
            on_fit_config_fn=lambda rnd: {"proximal_mu": proximal_mu},
        )
        return ServerAppComponents(strategy=strategy, config=ServerConfig(num_rounds=num_rounds))

    client_app = ClientApp(client_fn=client_fn)
    server_app = ServerApp(server_fn=server_fn)

    start = time.time()
    run_simulation(
        server_app=server_app, client_app=client_app, num_supernodes=NUM_HOSPITALS,
        backend_config={"client_resources": CLIENT_RESOURCES},
    )
    elapsed = time.time() - start
    print(f"Run '{run_name}' finished in {elapsed:.1f}s over {len(round_history)-1} rounds.")

    # Final global model (last round's parameters, held in eval_model_holder) -> test set
    test_metrics = evaluate_model(eval_model_holder["model"], pooled_test_loader)
    print(f"Final TEST metrics for '{run_name}': acc={test_metrics['accuracy']:.4f} "
          f"f1_macro={test_metrics['f1_macro']:.4f}")

    # Communication cost estimate
    per_round_mb = transmitted_size_mb(init_arrs) * NUM_HOSPITALS * 2  # up + down per client
    total_comm_mb = per_round_mb * len(round_history)

    return {
        "run_name": run_name, "data_source": data_source, "proximal_mu": proximal_mu,
        "round_history": round_history, "test_metrics": test_metrics,
        "elapsed_sec": elapsed, "total_comm_mb": total_comm_mb,
    }

## 6. Run All Four Experiments

This is the long-running cell. With `NUM_ROUNDS=8`, `LOCAL_EPOCHS=2`, and 3
CPU-bound Ray client actors, expect this to take a while even on a GPU
runtime (client training itself runs on CPU per client by default — see the
`CLIENT_NUM_GPUS` note in Section 0). Lower `NUM_ROUNDS` first if you just
want to sanity-check that everything runs before committing to a full run.

In [ ]:
experiment_configs = [
    ("FedAvg_nonIID",     "non_iid", 0.0),
    ("FedProx_mu0.01_nonIID", "non_iid", 0.01),
    ("FedProx_mu0.1_nonIID",  "non_iid", 0.1),
    ("FedAvg_IID",        "iid",     0.0),
]

all_results = {}
for run_name, data_source, mu in experiment_configs:
    all_results[run_name] = run_federated_experiment(run_name, data_source, mu)

# Persist raw results (minus in-memory model objects) for later reuse
serializable_results = {
    name: {k: v for k, v in r.items()} for name, r in all_results.items()
}
with open(os.path.join(OUTPUT_DIR, "federated_results.json"), "w") as f:
    json.dump(serializable_results, f, indent=2)
print("\nAll experiments complete. Results saved.")

## 7. Convergence Curves — All Strategies Overlaid

In [ ]:
plt.figure(figsize=(9, 5))
for run_name, r in all_results.items():
    hist_df = pd.DataFrame(r["round_history"])
    plt.plot(hist_df["round"], hist_df["val_f1_macro"], marker="o", label=run_name)

plt.axhline(y=CENTRALIZED_BASELINE["f1_macro"], color="black", linestyle="--",
            label=f"Centralized baseline ({CENTRALIZED_BASELINE['f1_macro']:.3f})")
plt.xlabel("Communication Round")
plt.ylabel("Validation F1 (macro)")
plt.title("Federated Convergence: FedAvg vs FedProx, IID vs Non-IID")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "convergence_curves.png"), dpi=150)
plt.show()

## 8. Final Comparison Table

In [ ]:
rows = []
for run_name, r in all_results.items():
    tm = r["test_metrics"]
    rows.append({
        "run": run_name,
        "data_split": r["data_source"],
        "proximal_mu": r["proximal_mu"],
        "test_accuracy": round(tm["accuracy"], 4),
        "test_f1_macro": round(tm["f1_macro"], 4),
        "test_f1_weighted": round(tm["f1_weighted"], 4),
        "gap_vs_centralized_f1": round(CENTRALIZED_BASELINE["f1_macro"] - tm["f1_macro"], 4),
        "wall_time_sec": round(r["elapsed_sec"], 1),
        "total_comm_mb": round(r["total_comm_mb"], 2),
    })

comparison_df = pd.DataFrame(rows).sort_values("test_f1_macro", ascending=False)
print(comparison_df.to_string(index=False))
comparison_df.to_csv(os.path.join(OUTPUT_DIR, "federated_comparison.csv"), index=False)

print(f"\nCentralized baseline (Phase 3, for reference): "
      f"accuracy={CENTRALIZED_BASELINE['accuracy']:.4f}, f1_macro={CENTRALIZED_BASELINE['f1_macro']:.4f}")
print("\nInterpretation notes:")
print("- Compare FedAvg_nonIID vs FedAvg_IID: the gap between them is the real")
print("  cost of non-IID data specifically (holding the aggregation algorithm fixed).")
print("- Compare FedAvg_nonIID vs the FedProx runs: if FedProx converges faster")
print("  or reaches higher F1 under non-IID, that's evidence the proximal term is")
print("  doing its job of limiting client drift.")
print("- 'gap_vs_centralized_f1' is the accuracy cost of federating at all,")
print("  vs. having all data in one place (Phase 3).")

## 9. Confusion Matrix — Best Federated Run

In [ ]:
best_run_name = comparison_df.iloc[0]["run"]
best_run = all_results[best_run_name]
tm = best_run["test_metrics"]
print(f"Best federated run: {best_run_name}")
print(classification_report(tm["labels"], tm["preds"], target_names=CATEGORIES, zero_division=0))

cm = confusion_matrix(tm["labels"], tm["preds"])
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", xticklabels=CATEGORIES, yticklabels=CATEGORIES)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title(f"Confusion Matrix — Best Federated Run ({best_run_name})")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "federated_confusion_matrix.png"), dpi=150)
plt.show()

## 10. What to Look For in Your Results

- **FedAvg_IID should be your best or near-best result and should converge
  fastest.** If it isn't, something about the setup (learning rate, local
  epochs) may need tuning independent of the non-IID question — diagnose that
  before drawing conclusions about non-IID/FedProx effects.
- **FedAvg_nonIID vs. FedAvg_IID gap** is your headline "cost of non-IID data"
  finding for the report.
- **Whether FedProx actually helps** (either μ) over plain FedAvg on the
  non-IID split is genuinely an open empirical question for your specific
  data — don't assume it must win; report whichever way it actually goes,
  since a null result here ("FedProx didn't meaningfully help at this scale")
  is still a valid, reportable finding, especially with only 3 clients where
  client drift has less room to compound than in larger federations.
- **`gap_vs_centralized_f1`** across all four runs tells you the overall
  federation cost — a small gap (a few points of F1) is the standard, expected
  result and a genuinely good outcome to report; it demonstrates federation
  approximates centralized performance while keeping data local.